# Celebal Technologies Summer Internship 2026
## Pipeline Orchestration & Validation
### FMCG Data Consolidation & Analytics Platform


**Objective:** Orchestrate the complete Bronze → Silver → Gold pipeline and validate results.

**Steps:**
1. Run complete pipeline end-to-end
2. Validate data quality at each layer
3. Generate pipeline execution report

In [0]:

# PIPELINE ORCHESTRATION
# End-to-End Pipeline Runner

from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.functions import sum as spark_sum, count, round as spark_round
import time

print("=" * 65)
print("  FMCG DATA PIPELINE - ORCHESTRATION STARTED")
print("=" * 65)

pipeline_start = time.time()


# Step 1: Validate Bronze Layer

print("\n BRONZE LAYER VALIDATION")
print("-" * 40)

bronze_a = spark.read.table("bronze_fmcg.company_a_raw")
bronze_b = spark.read.table("bronze_fmcg.company_b_raw")

print(f" Company A (Superstore ERP):  {bronze_a.count():,} rows")
print(f" Company B (Legacy FMCG):         {bronze_b.count():,} rows")
print(f" Total Bronze Records:        {bronze_a.count() + bronze_b.count():,} rows")

  FMCG DATA PIPELINE - ORCHESTRATION STARTED

 BRONZE LAYER VALIDATION
----------------------------------------
 Company A (Superstore ERP):  9,994 rows
 Company B (Legacy FMCG):         15 rows
 Total Bronze Records:        10,009 rows


In [0]:

# Step 2: Validate Silver Layer

print("\n SILVER LAYER VALIDATION")
print("-" * 40)

silver = spark.read.table("silver_fmcg.unified_sales")
silver_count = silver.count()

# Check nulls
from pyspark.sql.functions import when
null_count = silver.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0))
    for c in silver.columns
]).collect()[0]
total_nulls = sum(null_count)

# Check duplicates
distinct_count = silver.dropDuplicates(["order_id", "product_name"]).count()
duplicates = silver_count - distinct_count

print(f" Unified dataset:     {silver_count:,} rows")
print(f" Null values:         {total_nulls} (handled)")
print(f" Duplicates:          {duplicates} (removed)")
print(f" Company A records:   {silver.filter(col('source_company') == 'Company_A').count():,}")
print(f" Company B records:   {silver.filter(col('source_company') == 'Company_B').count():,}")
silver.show(5)


 SILVER LAYER VALIDATION
----------------------------------------
 Unified dataset:     10,001 rows
 Null values:         0 (handled)
 Duplicates:          0 (removed)
 Company A records:   9,986
 Company B records:   15
+--------------+----------+----------+-----------+---------------+--------+-------+------------+-------------+---------------+------------+--------------------+-------+--------+--------+-------+--------------+
|      order_id|order_date| ship_date|customer_id|  customer_name| Segment| Region|        city|      Country|       Category|sub_category|        product_name|  Sales|Quantity|Discount| Profit|source_company|
+--------------+----------+----------+-----------+---------------+--------+-------+------------+-------------+---------------+------------+--------------------+-------+--------+--------+-------+--------------+
|CA-2014-115812|2014-06-09|2014-06-14|   BH-11710|Brosina Hoffman|CONSUMER|   WEST| Los Angeles|United States|     Technology|      Phones|Mitel 532

# Pipeline Orchestration Summary
## FMCG Data Consolidation & Analytics Platform

---

## Pipeline Architecture
Raw Sources → Bronze Layer → Silver Layer → Gold Layer → BI/Analytics
│              │              │              │
CSV Files    Delta Tables   Cleaned &      Star Schema
Legacy DB    (Raw Data)    Standardized    + KPI Tables
APIs                        Unified Data

---

## Execution Results

###  Bronze Layer
| Source | System | Records |
|--------|--------|---------|
| Company A | Superstore ERP | 9,994 |
| Company B | Legacy FMCG System | 15 |
| **Total** | | **10,009** |

###  Silver Layer
| Operation | Result |
|-----------|--------|
| Null values handled | 301 values |
| Duplicates removed | 8 rows |
| Schema standardized | yes |
| Final unified records | 10,001 |

###  Gold Layer (Star Schema)
| Table | Type | Rows |
|-------|------|------|
| fact_sales | Fact | 10,001 |
| dim_customer | Dimension | 804 |
| dim_product | Dimension | 1,862 |
| dim_region | Dimension | 593 |
| dim_time | Dimension | 1,252 |
| kpi_sales_by_region | KPI | 8 |
| kpi_product_performance | KPI | 1,862 |
| kpi_customer_lifetime_value | KPI | 804 |
| kpi_company_comparison | KPI | 2 |

---

## Key Business Insights

### Sales by Region
| Region | Total Sales | Profit | Orders |
|--------|------------|--------|--------|
| WEST | ₹7,13,369 | ₹1,07,289 | 3,202 |
| EAST | ₹6,71,319 | ₹91,434 | 2,845 |
| CENTRAL | ₹4,97,800 | ₹40,150 | 2,323 |
| SOUTH | ₹3,88,269 | ₹46,450 | 1,616 |

### Top Products
| Product | Sales | Profit |
|---------|-------|--------|
| Canon imageCLASS | ₹61,599 | ₹25,199 |
| Fellowes PB500 | ₹27,453 | ₹7,753 |
| Cisco TelePresence | ₹22,638 | -₹1,811 |

### Company Performance
| Company | Total Sales | Profit | Orders |
|---------|------------|--------|--------|
| Company A | ₹22,70,758 | ₹2,85,324 | 9,986 |
| Company B | ₹39,400 | ₹5,867 | 15 |

---

## Technology Stack
| Component | Technology |
|-----------|-----------|
| Platform | Databricks Community Edition |
| Processing | Apache Spark 4.1.0 (PySpark) |
| Storage | Delta Lake |
| Architecture | Medallion (Bronze/Silver/Gold) |
| Data Model | Star Schema |

---

## Pipeline Status
| Metric | Value |
|--------|-------|
| Total sources | 2 companies |
| Bronze records | 10,009 |
| Silver records | 10,001 |
| Gold tables | 9 |
| KPI tables | 4 |
| Data quality |  Validated |
| Pipeline status | **SUCCESS ** |